# Notebook 2 — Book-level aggregation & derived indices
**Stage 10 (Correlation/Stats prep)**

This notebook takes the **topic-level lookup table** produced in Notebook 1 and joins it to **book topic mixture data** to produce:

- book-level taxonomy proportions (long + wide)
- optional segment-level taxonomy proportions (if chapter/segment topic probs are available)
- derived **taxonomy-proxy indices** aligned to your hypotheses

All paths are encoded the same way as in Notebook 1 (project_root + results/...).


In [8]:
# --- Robust project_root fix: handle not defined and relative/invalid cases safely ---
from pathlib import Path

def safe_project_root(project_root_var=None) -> Path:
    """Ensure project_root is defined, exists, and is absolute.
    Attempts to infer sensible location if not provided or not valid."""
    # 1. Use arg if present, else try global, else fallback to cwd scan
    _pr = project_root_var
    try:
        if _pr is None:
            _pr = globals().get("project_root", None)
        # If still None or empty, try environment
        if _pr in (None, ""):
            _pr = Path.home()  # fallback to home, for now
        else:
            _pr = Path(_pr)
    except Exception:
        _pr = Path.cwd()
    
    _pr = _pr.expanduser().resolve()
    # If not valid, try scanning for src/results
    SEARCH_MARKERS = ["src", "results"]
    def looks_like_root(p):
        return all((p/pth).exists() for pth in SEARCH_MARKERS)
    
    # If path is ".", doesn't exist, or doesn't have src/results, look up tree
    if not _pr.exists() or _pr == Path(".") or not looks_like_root(_pr):
        # 1. Check up from cwd
        for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
            if looks_like_root(p):
                _pr = p
                break
        else:
            # 2. Fallback to hardcoded known path (edit if needed)
            KNOWN = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
            if looks_like_root(KNOWN):
                _pr = KNOWN.resolve()
            else:
                raise RuntimeError("Could not determine a valid project_root!")
    return _pr

project_root = safe_project_root()
print(f"✓ Project root: {project_root}")

✓ Project root: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor


In [9]:
from __future__ import annotations

import os
import json
from pathlib import Path
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd

# ---- Load defaults from your project (if available) ----
try:
    from src.stage06_topic_exploration.explore_retrained_model import (
        DEFAULT_BASE_DIR,
        DEFAULT_EMBEDDING_MODEL,
    )
except Exception:
    DEFAULT_BASE_DIR = project_root / "models"
    DEFAULT_EMBEDDING_MODEL = "paraphrase-MiniLM-L6-v2"

print(f"✓ Project root: {project_root}")
print(f"DEFAULT_BASE_DIR: {DEFAULT_BASE_DIR}")
print(f"DEFAULT_EMBEDDING_MODEL: {DEFAULT_EMBEDDING_MODEL}")

# ---- Notebook 1 output directory (must match exactly) ----
OUT_DIR_STAGE10 = project_root / "results" / "stage10_correlation_analysis"
EDA_DIR = OUT_DIR_STAGE10 / "taxonomy_radway_eda"     # Notebook 1 output folder
NB2_DIR = OUT_DIR_STAGE10 / "notebook2_book_features" # Notebook 2 output folder
NB2_DIR.mkdir(parents=True, exist_ok=True)

TOPIC_LOOKUP_PATH = EDA_DIR / "topic_lookup.parquet"  # produced by Notebook 1

print(f"EDA_DIR:  {EDA_DIR}")
print(f"NB2_DIR:  {NB2_DIR}")
print(f"TOPIC_LOOKUP_PATH: {TOPIC_LOOKUP_PATH}")


✓ Project root: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
DEFAULT_BASE_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models
DEFAULT_EMBEDDING_MODEL: paraphrase-MiniLM-L6-v2
EDA_DIR:  /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda
NB2_DIR:  /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/notebook2_book_features
TOPIC_LOOKUP_PATH: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/topic_lookup.parquet


In [ ]:
def first_existing(paths: List[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None

def glob_first(base: Path, patterns: List[str]) -> Optional[Path]:
    for pat in patterns:
        matches = sorted(base.glob(pat))
        if matches:
            return matches[0]
    return None

def find_inputs(project_root: Path) -> Dict[str, Optional[Path]]:
    """Best-effort discovery of required inputs.
    We prefer your pipeline outputs if they exist; otherwise fall back to raw contracts."""
    candidates = {}

    # 1) Book topic mixtures (topic_id, prob per book)
    # Primary source: generate_topic_probabilities.py outputs in stage10_correlation_analysis
    # Typical names in your docs: book_topic_probs.csv
    # Can also be derived from sentence_df_with_topics.parquet
    candidates['book_topic_probs'] = first_existing([
        project_root / "results" / "stage10_correlation_analysis" / "book_topic_probs.parquet",  # Primary: output from generate_topic_probabilities.py
        project_root / "results" / "stage10_correlation_analysis" / "book_topic_probs.csv",
        project_root / "book_topic_probs.csv",
        project_root / "data" / "book_topic_probs.csv",
        project_root / "results" / "book_topic_probs.csv",
    ]) or glob_first(project_root / "results", [
        "**/book_topic_probs.csv",
        "**/book_topic_probs.parquet",
        "**/book_topic_proportions*.parquet",
        "**/book_topic_proportions*.csv",
    ])

    # 2) Book metadata (Goodreads, rating_class / group, author_id, length, year)
    # Can be found in goodreads.csv or extracted from sentence_df_with_topics.parquet
    candidates['books_meta'] = first_existing([
        project_root / "books_meta.csv",
        project_root / "data" / "books_meta.csv",
        project_root / "results" / "books_meta.csv",
        project_root / "data" / "processed" / "goodreads.csv",  # Standard pipeline location
    ]) or glob_first(project_root / "results", [
        "**/books_meta.csv",
        "**/books_meta.parquet",
        "**/books_metadata*.csv",
        "**/books_metadata*.parquet",
    ])

    # 3) Optional: chapter/segment topic probs
    # Primary source: generate_topic_probabilities.py outputs in stage10_correlation_analysis
    candidates['chapter_topic_probs'] = first_existing([
        project_root / "results" / "stage10_correlation_analysis" / "chapter_topic_probs.parquet",  # Primary: output from generate_topic_probabilities.py
        project_root / "results" / "stage10_correlation_analysis" / "chapter_topic_probs.csv",
        project_root / "chapter_topic_probs.csv",
        project_root / "data" / "chapter_topic_probs.csv",
        project_root / "results" / "chapter_topic_probs.csv",
    ]) or glob_first(project_root / "results", [
        "**/chapter_topic_probs.csv",
        "**/chapter_topic_probs.parquet",
        "**/segment_topic_probs*.csv",
        "**/segment_topic_probs*.parquet",
    ])

    # 4) If Stage09 already computed category proportions, use those directly
    candidates['stage09_book_category_props'] = first_existing([
        project_root / "results" / "stage09_category_mapping" / "stage2_theory_driven_categories" / "book_category_proportions.parquet",
        project_root / "results" / "stage09_category_mapping" / "stage2_theory_driven_categories" / "book_category_proportions.csv",
    ])

    # 5) Sentence dataframe with topics (can be used to derive book_topic_probs and books_meta)
    candidates['sentence_df_with_topics'] = first_existing([
        project_root / "data" / "processed" / "sentence_df_with_topics.parquet",
        project_root / "data" / "processed" / "sentence_df_with_topics.csv",
        project_root / "data" / "sentence_df_with_topics.parquet",
    ]) or glob_first(project_root / "data", [
        "**/sentence_df_with_topics.parquet",
        "**/sentence_df_with_topics.csv",
    ])

    # 6) Goodreads metadata (alternative source for books_meta)
    candidates['goodreads_csv'] = first_existing([
        project_root / "data" / "processed" / "goodreads.csv",
        project_root / "data" / "goodreads.csv",
    ])

    return candidates

inputs = find_inputs(project_root)

# Derive books_meta from sentence dataframe if available (has correct book_id format)
if inputs.get('books_meta') is None and inputs.get('sentence_df_with_topics') is not None:
    try:
        import pandas as pd
        sent_df = pd.read_parquet(inputs['sentence_df_with_topics'])
        # Extract unique book-level metadata
        book_cols = ['book_id']
        meta_cols = ['rating_class', 'rating_mean', 'rating_count', 'Author', 'Book Title']
        available_meta = [c for c in meta_cols if c in sent_df.columns]
        if available_meta:
            books_meta = sent_df[['book_id'] + available_meta].drop_duplicates(subset=['book_id'])
            # Save temporarily so it can be loaded later
            temp_meta_path = NB2_DIR / "books_meta_from_sentence_df.parquet"
            books_meta.to_parquet(temp_meta_path, index=False)
            inputs['books_meta'] = temp_meta_path
            print(f"✓ Derived books_meta from sentence_df_with_topics: {temp_meta_path}")
    except Exception as e:
        print(f"⚠ Could not derive books_meta from sentence_df: {e}")

# Fallback to goodreads.csv if still not found
if inputs.get('books_meta') is None and inputs.get('goodreads_csv') is not None:
    # Try to use goodreads.csv as books_meta source
    inputs['books_meta'] = inputs['goodreads_csv']
    print(f"⚠ books_meta not found, will use goodreads.csv: {inputs['goodreads_csv']}")
    print("  Note: You may need to create book_id from Author+Title to match other data")

print("\nDiscovered inputs:")
for k,v in inputs.items():
    print(f"  {k}: {v}")



Discovered inputs:
  book_topic_probs: None
  books_meta: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/data/processed/goodreads.csv
  chapter_topic_probs: None
  stage09_book_category_props: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage09_category_mapping/stage2_theory_driven_categories/book_category_proportions.parquet
  sentence_df_with_topics: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/data/processed/sentence_df_with_topics.parquet
  goodreads_csv: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/data/processed/goodreads.csv


In [11]:
# --- Path resolution report (run this first if anything fails) ---
from pathlib import Path
import os, time

def describe(p: Path | None, label: str):
    if p is None:
        print(f"{label:<28} : None")
        return
    p = Path(p)
    exists = p.exists()
    kind = 'dir' if exists and p.is_dir() else 'file' if exists else 'missing'
    size = p.stat().st_size if exists and p.is_file() else None
    mtime = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(p.stat().st_mtime)) if exists else None
    print(f"{label:<28} : {p}")
    print(f"  exists={exists}  type={kind}  size={size}  mtime={mtime}")

print("\n" + "="*90)
print("NOTEBOOK 2 — PATH REPORT")
print("="*90)
print(f"CWD: {Path.cwd()}")
print(f"PROJECT_ROOT: {project_root}")

# Notebook 1 artifacts
describe(EDA_DIR, 'EDA_DIR (nb1 output)')
describe(TOPIC_LOOKUP_PATH, 'topic_lookup.parquet')

# Discovered inputs (best-effort)
print("\nDiscovered inputs (from find_inputs):")
for k, v in inputs.items():
    describe(v, k)

# Pipeline-expected canonical locations (if you want to enforce them)
print("\nCanonical pipeline paths (expected if you ran full pipeline):")
canon_stage09 = project_root / 'results' / 'stage09_category_mapping' / 'stage2_theory_driven_categories' / 'book_category_proportions.parquet'
canon_stage09_csv = project_root / 'results' / 'stage09_category_mapping' / 'stage2_theory_driven_categories' / 'book_category_proportions.csv'
describe(canon_stage09, 'stage09 book_category_proportions.parquet')
describe(canon_stage09_csv, 'stage09 book_category_proportions.csv')

print("\nNotebook 2 outputs:")
describe(NB2_DIR, 'NB2_DIR')
print("="*90)



NOTEBOOK 2 — PATH REPORT
CWD: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/notebooks/07_analysis/statistical_analysis
PROJECT_ROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
EDA_DIR (nb1 output)         : /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda
  exists=True  type=dir  size=None  mtime=2025-12-14 23:19:28
topic_lookup.parquet         : /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/topic_lookup.parquet
  exists=True  type=file  size=49269  mtime=2025-12-14 23:19:28

Discovered inputs (from find_inputs):
book_topic_probs             : None
books_meta                   : /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/data/processed/goo

In [12]:
assert TOPIC_LOOKUP_PATH.exists(), (
    "topic_lookup.parquet not found. Run Notebook 1 first, or verify your project_root points at the repo root.\n"
    f"Expected: {TOPIC_LOOKUP_PATH}"
)

topic_lookup = pd.read_parquet(TOPIC_LOOKUP_PATH)

print("topic_lookup shape:", topic_lookup.shape)
display(topic_lookup.head())

# Basic QA: required columns
required_cols = {'topic_id','taxonomy_main_id','taxonomy_main_name','taxonomy_main_group'}
missing = required_cols - set(topic_lookup.columns)
assert not missing, f"topic_lookup missing required columns: {missing}"


topic_lookup shape: (368, 20)


,topic_id,taxonomy_main_id,taxonomy_main_name,taxonomy_main_group,taxonomy_secondary_id,taxonomy_secondary_name,taxonomy_secondary_group,taxonomy_confidence,taxonomy_is_noise,radway_main_id,radway_main_name,radway_phase,radway_phase_name,radway_is_none,radway_confidence,label,scene_summary,primary_categories,secondary_categories,label_is_noise
0,0,4.3,"Secrets, Misunderstandings & Hidden Information",Relationship Trajectory (Main Couple),None,None,None,medium,False,R2,Heroine reacts antagonistically to the hero,I,Initial Conflict & Isolation,False,medium,Negotiating Deal,The couple discusses terms and expectations fo...,"relationship_conflict, domestic_life","setting:living_room, activity:discussion",False
1,1,2.3,Explicit Sexual Acts,"Sexuality, Attraction & Intimacy",None,None,None,high,False,R12,Heroine responds sexually and emotionally,III,Commitment & Restoration,False,high,Intimate Breast And Nipple Kissing,He gently kisses her breasts and nipples while...,"physical_affection, sexual_content","setting:bedroom, activity:kissing",False
2,2,2.3,Explicit Sexual Acts,"Sexuality, Attraction & Intimacy",None,None,None,high,False,R12,Heroine responds sexually and emotionally,III,Commitment & Restoration,False,high,Clitoral Stimulation During Foreplay,"She spreads her legs, allowing him to run his ...","sexual_content, romance_core","setting:bedroom, activity:oral_sex, sexual:cli...",False
3,3,4.2,"Bonding, Everyday Intimacy & Growth",Relationship Trajectory (Main Couple),None,None,None,high,False,R8,Hero treats heroine tenderly,II,Turning Point & Recognition,False,medium,Dinner Invitation,"He invites her to dinner, suggesting a restaur...","romance_core, social_setting","setting:restaurant, activity:invitation",False
4,4,4.3,"Secrets, Misunderstandings & Hidden Information",Relationship Trajectory (Main Couple),None,None,None,medium,False,R7,Hero and heroine are physically or emotionally...,I,Initial Conflict & Isolation,False,medium,Unclear Relationship Feelings,The couple expresses confusion about their rel...,"romance_core, relationship_conflict","setting:home, activity:conversation",False


In [13]:
def load_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in (".parquet", ".pq"):
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {path}")

book_cat_long = None
source_used = None

if inputs.get('stage09_book_category_props') is not None:
    p = inputs['stage09_book_category_props']
    book_cat_long = load_table(p)
    source_used = f"stage09_book_category_props: {p}"
    print(f"✓ Using Stage09 book category proportions: {p}")
else:
    p = inputs.get('book_topic_probs')
    assert p is not None and Path(p).exists(), (
        "Could not find book_topic_probs.csv/parquet.\n"
        "Looked under project_root and results/**.\n"
        "If your file name differs, edit the discovery patterns in find_inputs()."
    )
    book_topic = load_table(Path(p))
    source_used = f"book_topic_probs: {p}"
    print(f"✓ Using book topic probs: {p}")
    display(book_topic.head())

    # Normalize column names
    # expected: book_id, topic_id, prob
    col_map = {c.lower(): c for c in book_topic.columns}
    # find likely columns
    def pick(name_options):
        for opt in name_options:
            if opt in col_map:
                return col_map[opt]
        return None

    c_book = pick(['book_id','book','bookid'])
    c_topic = pick(['topic_id','topic','topicid'])
    c_prob = pick(['prob','probability','topic_prob','weight'])

    assert c_book and c_topic and c_prob, (
        f"book_topic_probs columns not recognized. Found: {list(book_topic.columns)}\n"
        "Need columns like: book_id, topic_id, prob"
    )

    book_topic = book_topic.rename(columns={c_book:'book_id', c_topic:'topic_id', c_prob:'prob'})
    book_topic['topic_id'] = book_topic['topic_id'].astype(int, errors='ignore')
    book_topic['prob'] = pd.to_numeric(book_topic['prob'], errors='coerce')

    # Join to taxonomy main categories using topic_lookup
    book_topic = book_topic.merge(
        topic_lookup[['topic_id','taxonomy_main_id','taxonomy_main_name','taxonomy_main_group']],
        on='topic_id', how='left'
    )

    # Track unmapped probability mass per book (important QA)
    book_topic['is_mapped'] = book_topic['taxonomy_main_id'].notna()
    unmapped_mass = (book_topic
                     .assign(unmapped_prob=lambda d: np.where(d['is_mapped'], 0.0, d['prob']))
                     .groupby('book_id', as_index=False)['unmapped_prob'].sum()
                     .rename(columns={'unmapped_prob':'unmapped_topic_mass'}))

    # Aggregate to book x taxonomy_main_id (long format)
    book_cat_long = (book_topic[book_topic['is_mapped']]
        .groupby(['book_id','taxonomy_main_id','taxonomy_main_name','taxonomy_main_group'], as_index=False)['prob']
        .sum()
        .rename(columns={'prob':'category_prop_raw'})
    )

    # Normalize within book across mapped categories so proportions sum to 1 (mapped space)
    totals = book_cat_long.groupby('book_id', as_index=False)['category_prop_raw'].sum().rename(columns={'category_prop_raw':'mapped_mass'})
    book_cat_long = book_cat_long.merge(totals, on='book_id', how='left')
    book_cat_long['category_prop'] = book_cat_long['category_prop_raw'] / book_cat_long['mapped_mass']

    # Attach unmapped mass
    book_cat_long = book_cat_long.merge(unmapped_mass, on='book_id', how='left')

print("\nSource used:", source_used)
print("book_cat_long shape:", book_cat_long.shape)
display(book_cat_long.head())


✓ Using Stage09 book category proportions: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage09_category_mapping/stage2_theory_driven_categories/book_category_proportions.parquet

Source used: stage09_book_category_props: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage09_category_mapping/stage2_theory_driven_categories/book_category_proportions.parquet
book_cat_long shape: (2215, 6)


,book_id,rating_class,main_category_id,n_sentences,total_sentences,prop
0,15197.0,good,1.1,4,1327,0.003014
1,15197.0,good,1.5,1,1327,0.000754
2,15197.0,good,2.1,56,1327,0.042200
3,15197.0,good,2.2,152,1327,0.114544
4,15197.0,good,2.3,206,1327,0.155237


In [7]:
books_meta = None
if inputs.get('books_meta') is not None:
    meta_path = Path(inputs['books_meta'])
    books_meta = load_table(meta_path)
    print(f"✓ Loaded books_meta: {meta_path}  shape={books_meta.shape}")
    display(books_meta.head())

    # Standardize common columns
    # group/rating_class
    if 'rating_class' not in books_meta.columns and 'group' in books_meta.columns:
        books_meta = books_meta.rename(columns={'group':'rating_class'})

    # author_id normalization
    if 'author_id' not in books_meta.columns:
        # attempt fallbacks
        for alt in ['author','authorid','author_id']:
            if alt in books_meta.columns:
                books_meta = books_meta.rename(columns={alt:'author_id'})
                break

    # book_id normalization (ID from goodreads.csv should be renamed to book_id)
    if 'book_id' not in books_meta.columns:
        # attempt fallbacks
        for alt in ['ID', 'id', 'book_id', 'bookid']:
            if alt in books_meta.columns:
                books_meta = books_meta.rename(columns={alt:'book_id'})
                break
    
    # Ensure book_id types match (convert to float to match book_cat_long)
    if 'book_id' in books_meta.columns:
        books_meta['book_id'] = pd.to_numeric(books_meta['book_id'], errors='coerce')

    # merge
    book_cat_long = book_cat_long.merge(
        books_meta,
        on='book_id', how='left', suffixes=('', '_meta')
    )
else:
    print("⚠ No books_meta found. book_cat_long will not include rating_class/controls unless already present.")

# Ensure rating_class exists if possible
if 'rating_class' in book_cat_long.columns:
    print("rating_class distribution:")
    print(book_cat_long['rating_class'].value_counts(dropna=False).head(10))


✓ Loaded books_meta: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/data/processed/goodreads.csv  shape=(97, 14)


,ID,Author,Title,URL,SeriesName,Summary,Genres,Score,RatingsCount,ReviewsCount,Pages,PublishedDate,Popularity_ReadingNow,Popularity_Wishlisted
0,35053870,sarina bowen,brooklynaire,https://www.goodreads.com/book/show/35053870-b...,Brooklyn Bruisers #4,"You’d think a billion dollars, a professional ...","Romance, Sports, Sports Romance, Contemporary,...",4.07,20705,2322,298,2018-02-12,2261,13100
1,28869598,sarina bowen,hard hitter,https://www.goodreads.com/book/show/28869598-h...,Brooklyn Bruisers #2,"He’s a fighter in the rink, but he’s about to ...","Romance, Sports, Sports Romance, Contemporary,...",4.05,10818,1049,336,2017-01-03,495,5907
2,30627346,sarina bowen,pipe dreams,https://www.goodreads.com/book/show/30627346-p...,Brooklyn Bruisers #3,"A goalie has to trust his instincts, even when...","Romance, Sports, Sports Romance, Contemporary,...",3.92,9532,975,336,2017-05-02,348,5180
3,17561022,j. clare,stranded with a billionaire,https://www.goodreads.com/book/show/17561022-s...,Billionaire Boys Club #1,The Billionaire Boys Club is a secret society ...,"Romance, Contemporary Romance, Contemporary, E...",3.82,14878,1009,215,2013-04-16,335,8684
4,43728457,j. clare,beauty and the billionaire,https://www.goodreads.com/book/show/43728457-b...,Dirty Fairy Tales #1,Ruthless Bastard. White Knight. But I just cal...,"Romance, Contemporary, Contemporary Romance, E...",3.85,9954,691,336,2019-01-27,15700,7770


KeyError: 'book_id'

In [ ]:
# Save long format
book_cat_long_out = NB2_DIR / "book_taxonomy_main_props_long.parquet"
book_cat_long.to_parquet(book_cat_long_out, index=False)
print(f"✓ Saved: {book_cat_long_out}")

# Wide format for modeling convenience
book_cat_wide = (book_cat_long
                 .pivot_table(index='book_id', columns='taxonomy_main_id', values='category_prop', aggfunc='sum', fill_value=0.0)
                 .reset_index())
book_cat_wide_out = NB2_DIR / "book_taxonomy_main_props_wide.parquet"
book_cat_wide.to_parquet(book_cat_wide_out, index=False)
print(f"✓ Saved: {book_cat_wide_out}")

display(book_cat_wide.head())


In [ ]:
# -----------------------------
# Taxonomy-proxy indices
# -----------------------------
# These are computed using YOUR ACTUAL taxonomy_main_name values present in the final model.
# They are "proxy" indices until the A–S composite mapping is finalized.

# Helper: build per-book lookup from long table
# We'll use taxonomy_main_name as the human-facing component key; fall back to IDs if names missing.
book_cat_by_name = (book_cat_long
    .dropna(subset=['taxonomy_main_name'])
    .groupby(['book_id','taxonomy_main_name'], as_index=False)['category_prop']
    .sum()
)

# Convert to wide-by-name for easy index computation
wide_name = (book_cat_by_name.pivot_table(index='book_id', columns='taxonomy_main_name', values='category_prop', aggfunc='sum', fill_value=0.0))

# ---- Component sets (edit here if you want to tighten/loosen the mapping) ----
COMPONENTS = {
    # Approximate "love/commitment/tenderness"
    "commitment_hea": [
        "Reconciliation, Commitments & HEA",
    ],
    "bonding_growth": [
        "Bonding, Everyday Intimacy & Growth",
    ],
    "positive_emotions": [
        "Positive Emotions & Security",
    ],
    "nonexplicit_affection": [
        "Kissing & Non-Explicit Affection",
    ],
    # Sex
    "explicit": [
        "Explicit Sexual Acts",
    ],
    # Conflict / negativity
    "miscommunication": [
        "Secrets, Misunderstandings & Hidden Information",
    ],
    "neg_affect": [
        "Negative Emotions & Distress",
    ],
    "breakup_conflict": [
        "Conflict, Distance & Breakup Threats",
    ],
    "violence_threat": [
        "Violence, Threats & Coercion",
    ],
    # Luxury/work proxy
    "elite_work": [
        "Hero's Elite Work & Business World",
    ],
    "public_leisure": [
        "Public & Leisure Spaces",
    ],
    "domestic": [
        "Domestic Spaces & Routines",
    ],
}

def sum_components(wide: pd.DataFrame, names: list) -> pd.Series:
    present = [c for c in names if c in wide.columns]
    if not present:
        return pd.Series(0.0, index=wide.index)
    return wide[present].sum(axis=1)

# Build component columns
components_df = pd.DataFrame(index=wide_name.index)
for comp, names in COMPONENTS.items():
    components_df[comp] = sum_components(wide_name, names)

# ---- Indices aligned to hypotheses (taxonomy-proxy versions) ----
indices = pd.DataFrame(index=wide_name.index)
indices['love_over_sex'] = (
    components_df['commitment_hea']
    + components_df['bonding_growth']
    + components_df['positive_emotions']
    + components_df['nonexplicit_affection']
    - components_df['explicit']
)

indices['hea_index'] = (
    components_df['commitment_hea']
)

indices['explicitness_ratio'] = (
    components_df['explicit']
    / (components_df['explicit'] + components_df['commitment_hea'] + components_df['positive_emotions'] + components_df['nonexplicit_affection'] + 1e-9)
)

indices['dark_vs_tender'] = (
    (components_df['neg_affect'] + components_df['breakup_conflict'] + components_df['violence_threat'])
    - (components_df['positive_emotions'] + components_df['nonexplicit_affection'])
)

indices['miscommunication_balance'] = (
    (components_df['commitment_hea'] + components_df['bonding_growth'] + components_df['positive_emotions'])
    - components_df['miscommunication']
)

indices['luxury_saturation_proxy'] = (
    components_df['elite_work'] + components_df['public_leisure']
)

# Attach unmapped mass (if present)
unmapped = (book_cat_long[['book_id','unmapped_topic_mass']].drop_duplicates('book_id')
            .set_index('book_id'))
indices = indices.join(unmapped, how='left')

# Join metadata fields if available
if books_meta is not None:
    meta_cols = [c for c in ['rating_class','avg_rating','n_ratings','author_id','length_tokens','length_words','year'] if c in books_meta.columns]
    meta = books_meta[['book_id'] + meta_cols].drop_duplicates('book_id').set_index('book_id')
    indices = indices.join(meta, how='left')

indices = indices.reset_index()

indices_out = NB2_DIR / "indices_book_taxonomy_proxy.parquet"
indices.to_parquet(indices_out, index=False)
print(f"✓ Saved indices: {indices_out}")

display(indices.head())


In [ ]:
# Optional segment-level processing (begin/middle/end) if available
if inputs.get('chapter_topic_probs') is None:
    print("No chapter/segment topic probabilities found. Skipping segment-level aggregation.")
else:
    seg_path = Path(inputs['chapter_topic_probs'])
    seg = load_table(seg_path)
    print(f"✓ Loaded segment/chapter topic probs: {seg_path}  shape={seg.shape}")
    display(seg.head())

    # Normalize columns
    col_map = {c.lower(): c for c in seg.columns}
    def pick(name_options):
        for opt in name_options:
            if opt in col_map:
                return col_map[opt]
        return None

    c_book = pick(['book_id','book','bookid'])
    c_topic = pick(['topic_id','topic','topicid'])
    c_prob = pick(['prob','probability','topic_prob','weight'])
    c_seg  = pick(['segment','tert','tertile','chapter_segment','part','section'])

    # If no explicit segment column, we keep chapter_id and leave time-course to Notebook 6.
    if c_seg is None:
        print("⚠ No explicit segment column found (begin/middle/end). Segment analysis deferred to Notebook 6.")
    else:
        seg = seg.rename(columns={c_book:'book_id', c_topic:'topic_id', c_prob:'prob', c_seg:'segment'})
        seg['prob'] = pd.to_numeric(seg['prob'], errors='coerce')

        seg = seg.merge(
            topic_lookup[['topic_id','taxonomy_main_id','taxonomy_main_name','taxonomy_main_group']],
            on='topic_id', how='left'
        )
        seg['is_mapped'] = seg['taxonomy_main_id'].notna()

        seg_cat_long = (seg[seg['is_mapped']]
            .groupby(['book_id','segment','taxonomy_main_id','taxonomy_main_name','taxonomy_main_group'], as_index=False)['prob']
            .sum()
            .rename(columns={'prob':'category_prop_raw'})
        )
        totals = seg_cat_long.groupby(['book_id','segment'], as_index=False)['category_prop_raw'].sum().rename(columns={'category_prop_raw':'mapped_mass'})
        seg_cat_long = seg_cat_long.merge(totals, on=['book_id','segment'], how='left')
        seg_cat_long['category_prop'] = seg_cat_long['category_prop_raw'] / seg_cat_long['mapped_mass']

        seg_out = NB2_DIR / "segment_taxonomy_main_props_long.parquet"
        seg_cat_long.to_parquet(seg_out, index=False)
        print(f"✓ Saved segment taxonomy proportions: {seg_out}")
